# ESM-2 Embeddings - OPTIMIZED & PRODUCTION-READY
## Comparative Lassa-Ebola ML Model

**Phase 3: Generate 1,280D protein embeddings with optimizations**

### Key Improvements:
- ✅ Memory-efficient batch processing
- ✅ Checkpoint/resumable generation
- ✅ GPU acceleration support
- ✅ Comprehensive error handling
- ✅ Progress tracking with ETA
- ✅ Automatic fallback to CPU
- ✅ Mixed precision inference
- ✅ Output validation & statistics

## SECTION 0: Setup & Configuration

In [2]:
# System imports
import os
import sys
import json
import time
import gc
from pathlib import Path
from datetime import datetime

# Data science
import numpy as np
import pandas as pd
import torch
from Bio import SeqIO

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML & utilities
import esm
from tqdm import tqdm
from sklearn.decomposition import PCA
from scipy.spatial.distance import euclidean, cosine

print("✓ All imports successful")

✓ All imports successful


In [3]:
# Configuration
class Config:
    # Paths
    DATA_DIR = Path('data')
    EMBEDDING_DIR = DATA_DIR / 'embeddings'
    RESULTS_DIR = Path('results')
    FIGURES_DIR = RESULTS_DIR / 'figures'
    
    # Create directories
    for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    
    # Model settings
    MODEL_NAME = "esm2_t33_650M_UR50D"
    OUTPUT_DIM = 1280
    BATCH_SIZE = 16 if torch.cuda.is_available() else 4
    MAX_SEQ_LEN = 1022
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Checkpointing
    CHECKPOINT_INTERVAL = 100  # Save every 100 sequences
    
    # Logging
    LOG_FILE = RESULTS_DIR / 'embedding_log.txt'

print(f"Device: {Config.DEVICE}")
print(f"Batch size: {Config.BATCH_SIZE}")
print(f"Output dimension: {Config.OUTPUT_DIM}")

Device: cpu
Batch size: 4
Output dimension: 1280


In [4]:
# Logger
class Logger:
    def __init__(self, filepath):
        self.filepath = filepath
        self.log(f"\n{'='*70}\nExecution started: {datetime.now()}\n{'='*70}")
    
    def log(self, message):
        print(message)
        with open(self.filepath, 'a') as f:
            f.write(message + '\n')

logger = Logger(Config.LOG_FILE)
logger.log(f"Device: {Config.DEVICE}")
logger.log(f"Model: {Config.MODEL_NAME}")


Execution started: 2026-03-26 10:23:48.967414
Device: cpu
Model: esm2_t33_650M_UR50D


## SECTION 1: Load & Prepare Data

In [15]:
def load_fasta(filepath):
    """Load FASTA file with error handling"""
    sequences = []
    errors = []
    
    try:
        for record in SeqIO.parse(filepath, "fasta"):
            try:
                seq_str = str(record.seq).strip()
                if len(seq_str) > 0:
                    sequences.append({
                        'id': record.id,
                        'sequence': seq_str,
                        'length': len(seq_str)
                    })
            except Exception as e:
                errors.append((record.id, str(e)))
    except Exception as e:
        logger.log(f"Error reading {filepath}: {e}")
        return [], [(filepath, str(e))]
    
    if errors:
        logger.log(f"Skipped {len(errors)} sequences due to errors")
    
    return sequences, errors

logger.log("\n[LOADING DATA]")
lassa_seqs, lassa_errors = load_fasta('data/processed/lassa_cleaned.fasta')
ebola_seqs, ebola_errors = load_fasta('data/processed/ebola_cleaned.fasta')

logger.log(f"Lassa: {len(lassa_seqs)} sequences")
logger.log(f"Ebola: {len(ebola_seqs)} sequences")
logger.log(f"Total: {len(lassa_seqs) + len(ebola_seqs)} sequences")

# Combine
all_seqs = []
for s in lassa_seqs:
    s['virus'] = 'Lassa'
    all_seqs.append(s)
for s in ebola_seqs:
    s['virus'] = 'Ebola'
    all_seqs.append(s)

logger.log(f"Combined sequences: {len(all_seqs)}")


[LOADING DATA]
Lassa: 780 sequences
Ebola: 1610 sequences
Total: 2390 sequences
Combined sequences: 2390


## SECTION 2: Load ESM-2 Model

In [6]:
logger.log("\n[LOADING MODEL]")
logger.log(f"Loading {Config.MODEL_NAME}...")
logger.log("(This downloads ~1.3 GB on first run)\n")

try:
    model, alphabet = esm.pretrained.load_model_and_alphabet_hub(Config.MODEL_NAME)
    model = model.to(Config.DEVICE)
    model.eval()
    logger.log("✓ Model loaded successfully")
except Exception as e:
    logger.log(f"✗ Failed to load model: {e}")
    raise


[LOADING MODEL]
Loading esm2_t33_650M_UR50D...
(This downloads ~1.3 GB on first run)

✓ Model loaded successfully


## SECTION 3: Embedding Generation with Checkpointing

In [8]:
class EmbeddingGenerator:
    def __init__(self, model, alphabet, device, config):
        self.model = model
        self.alphabet = alphabet
        self.device = device
        self.config = config
        self.batch_converter = alphabet.get_batch_converter()
    
    def get_checkpoint_path(self, idx):
        return self.config.EMBEDDING_DIR / f"checkpoint_{idx:05d}.pt"
    
    def save_checkpoint(self, embeddings, idx):
        """Save partial results"""
        path = self.get_checkpoint_path(idx)
        torch.save(embeddings, path)
        return path
    
    def load_checkpoint(self, idx):
        """Load partial results"""
        path = self.get_checkpoint_path(idx)
        if path.exists():
            return torch.load(path)
        return None
    
    def process_batch(self, batch_seqs):
        """Process one batch of sequences"""
        try:
            # Truncate sequences
            batch_seqs = [s[:self.config.MAX_SEQ_LEN] for s in batch_seqs]
            
            # Prepare batch
            batch_labels = [f"seq_{i}" for i in range(len(batch_seqs))]
            batch_data = list(zip(batch_labels, batch_seqs))
            
            _, _, tokens = self.batch_converter(batch_data)
            tokens = tokens.to(self.device)
            
            # Generate embeddings
            with torch.no_grad():
                if self.device.type == 'cuda':
                    with torch.cuda.amp.autocast():
                        results = self.model(tokens, repr_layers=[33], return_contacts=False)
                else:
                    results = self.model(tokens, repr_layers=[33], return_contacts=False)
            
            token_embeddings = results["representations"][33]
            
            # Average across tokens
            batch_embeddings = []
            for j, seq in enumerate(batch_seqs):
                emb = token_embeddings[j, 1:len(seq)+1].mean(0)
                batch_embeddings.append(emb.cpu())
            
            return torch.stack(batch_embeddings)
        
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                logger.log(f"WARNING: OOM - reducing batch size")
                torch.cuda.empty_cache()
                # Try smaller batch
                if len(batch_seqs) > 1:
                    return torch.cat([
                        self.process_batch(batch_seqs[:len(batch_seqs)//2]),
                        self.process_batch(batch_seqs[len(batch_seqs)//2:])
                    ])
            raise
    
    def generate(self, sequences, resume=True):
        """Generate all embeddings with resumable checkpointing"""
        all_embeddings = []
        start_idx = 0
        
        # Try to resume
        if resume:
            for idx in range(0, len(sequences), self.config.CHECKPOINT_INTERVAL):
                checkpoint = self.load_checkpoint(idx)
                if checkpoint is not None:
                    all_embeddings.append(checkpoint)
                    start_idx = idx + len(checkpoint)
                else:
                    break
            
            if start_idx > 0:
                logger.log(f"Resuming from checkpoint at sequence {start_idx}")
        
        # Generate remaining
        pbar = tqdm(range(start_idx, len(sequences), self.config.BATCH_SIZE), 
                     desc="Embedding", unit="batch")
        
        for i in pbar:
            try:
                batch = sequences[i:i+self.config.BATCH_SIZE]
                batch_emb = self.process_batch(batch)
                all_embeddings.append(batch_emb)
                
                # Checkpoint
                if (i + self.config.BATCH_SIZE) % self.config.CHECKPOINT_INTERVAL == 0:
                    combined = torch.cat(all_embeddings)
                    self.save_checkpoint(combined, i)
                    pbar.set_postfix({'saved': f'{i+self.config.BATCH_SIZE}'})
            
            except Exception as e:
                logger.log(f"Error at batch {i}: {e}")
                raise
        
        return torch.cat(all_embeddings)

generator = EmbeddingGenerator(model, alphabet, Config.DEVICE, Config)

In [9]:
logger.log("\n[GENERATING EMBEDDINGS]")
logger.log(f"Sequences: {len(all_seqs):,}")
logger.log(f"Batch size: {Config.BATCH_SIZE}")
logger.log(f"Device: {Config.DEVICE}\n")

seq_strs = [s['sequence'] for s in all_seqs]

try:
    start_time = time.time()
    embeddings = generator.generate(seq_strs, resume=True)
    elapsed = time.time() - start_time
    
    logger.log(f"\n✓ Embeddings generated in {elapsed/60:.1f} minutes")
    logger.log(f"  Shape: {embeddings.shape}")
    logger.log(f"  Size: {embeddings.nbytes / (1024**3):.2f} GB")
    logger.log(f"  Speed: {len(embeddings)/elapsed:.1f} seq/sec")

except Exception as e:
    logger.log(f"\n✗ Error during embedding generation: {e}")
    import traceback
    logger.log(traceback.format_exc())
    raise


[GENERATING EMBEDDINGS]
Sequences: 2,390
Batch size: 4
Device: cpu



Embedding:   0%|                                     | 0/598 [00:12<?, ?batch/s]


KeyboardInterrupt: 

## SECTION 4: Validation & Statistics

In [ ]:
# ============================================================
# VALIDATION - AFTER CHECKPOINT CODE
# ============================================================

logger.log("\n[VALIDATION]")

# Convert embeddings to tensor if it's numpy
if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()
    logger.log("✓ Converted embeddings from NumPy to PyTorch tensor")

# Basic checks
assert embeddings.shape[0] == len(all_seqs), "Embedding count mismatch"
assert embeddings.shape[1] == Config.OUTPUT_DIM, "Embedding dimension mismatch"
assert not torch.isnan(embeddings).any(), "NaN values detected"
assert not torch.isinf(embeddings).any(), "Inf values detected"

logger.log("✓ Shape validation passed")
logger.log("✓ NaN/Inf check passed")

# Statistics
logger.log(f"\nEmbedding statistics:")
logger.log(f"  Min: {embeddings.min():.6f}")
logger.log(f"  Max: {embeddings.max():.6f}")
logger.log(f"  Mean: {embeddings.mean():.6f}")
logger.log(f"  Std: {embeddings.std():.6f}")
logger.log(f"  Dtype: {embeddings.dtype}")

logger.log("\n✓ All validations passed!")

## SECTION 5: Save Results

In [ ]:
logger.log("\n[SAVING]")

# Metadata
metadata = pd.DataFrame(all_seqs)
metadata['embedding_idx'] = range(len(metadata))
metadata_path = Config.EMBEDDING_DIR / 'metadata.csv'
metadata.to_csv(metadata_path, index=False)
logger.log(f"✓ Metadata: {metadata_path}")

# Embeddings
lassa_emb = embeddings[:len(lassa_seqs)].cpu().numpy()
ebola_emb = embeddings[len(lassa_seqs):].cpu().numpy()

npz_path = Config.EMBEDDING_DIR / 'embeddings.npz'
np.savez_compressed(npz_path, lassa=lassa_emb, ebola=ebola_emb, all=embeddings.cpu().numpy())
logger.log(f"✓ Embeddings: {npz_path}")
logger.log(f"  Lassa: {lassa_emb.shape}")
logger.log(f"  Ebola: {ebola_emb.shape}")

# PT format for PyTorch
pt_path = Config.EMBEDDING_DIR / 'embeddings.pt'
torch.save(embeddings, pt_path)
logger.log(f"✓ PyTorch format: {pt_path}")

## SECTION 6: Visualization

In [ ]:
logger.log("\n[VISUALIZATION]")

# PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings.numpy())

logger.log(f"PCA variance:")
logger.log(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
logger.log(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

lassa_2d = emb_2d[:len(lassa_seqs)]
ebola_2d = emb_2d[len(lassa_seqs):]

ax.scatter(lassa_2d[:, 0], lassa_2d[:, 1], c='#FF6B6B', label='Lassa', alpha=0.6, s=50)
ax.scatter(ebola_2d[:, 0], ebola_2d[:, 1], c='#4ECDC4', label='Ebola', alpha=0.6, s=50)

lassa_cent = lassa_2d.mean(axis=0)
ebola_cent = ebola_2d.mean(axis=0)

ax.scatter(*lassa_cent, c='darkred', marker='*', s=1000, edgecolor='black', linewidth=2)
ax.scatter(*ebola_cent, c='darkblue', marker='*', s=1000, edgecolor='black', linewidth=2)
ax.plot([lassa_cent[0], ebola_cent[0]], [lassa_cent[1], ebola_cent[1]], 'k--', linewidth=2, alpha=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
ax.set_title('PCA: ESM-2 Embeddings', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Config.FIGURES_DIR / 'embeddings_pca.png', dpi=300, bbox_inches='tight')
logger.log(f"✓ PCA plot: {Config.FIGURES_DIR / 'embeddings_pca.png'}")
plt.show()

## SECTION 7: Summary

In [ ]:
logger.log("\n" + "="*70)
logger.log("✅ COMPLETE: ESM-2 EMBEDDINGS GENERATED")
logger.log("="*70)

summary = {
    'timestamp': datetime.now().isoformat(),
    'total_sequences': len(all_seqs),
    'lassa_count': len(lassa_seqs),
    'ebola_count': len(ebola_seqs),
    'embedding_dim': Config.OUTPUT_DIM,
    'model': Config.MODEL_NAME,
    'device': str(Config.DEVICE),
    'generation_time_sec': elapsed,
    'embeddings_saved': str(pt_path),
    'metadata_saved': str(metadata_path),
}

summary_path = Config.RESULTS_DIR / 'embedding_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

logger.log(json.dumps(summary, indent=2))
logger.log("\n" + "="*70)

In [11]:
import os
import glob
import torch
import numpy as np
from tqdm import tqdm
import time

class CheckpointManager:
    """Resume embeddings from checkpoint"""
    
    def __init__(self, checkpoint_dir='data/embeddings'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
    
    def find_latest_checkpoint(self):
        checkpoints = sorted(glob.glob(f"{self.checkpoint_dir}/checkpoint_*.pt"))
        if checkpoints:
            latest = checkpoints[-1]
            seq_idx = int(latest.split('_')[-1].replace('.pt', ''))
            return latest, seq_idx
        return None, 0
    
    def load_checkpoint(self, checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        return {
            'embeddings': checkpoint['embeddings'],
            'processed_count': checkpoint['processed_count'],
            'seq_indices': checkpoint['seq_indices']
        }

In [14]:
# ============================================================
# CORRECT RESUME CODE - CONTINUE FROM 1500 SEQUENCES
# ============================================================

import os
import torch
import numpy as np
import glob
from tqdm import tqdm

print("=" * 70)
print("RESUMING EMBEDDING GENERATION - SMART CONTINUATION")
print("=" * 70)

# Step 1: Find the ACTUAL latest checkpoint with correct data
checkpoint_dir = 'data/embeddings'
checkpoints = sorted(glob.glob(f"{checkpoint_dir}/checkpoint_*.pt"))

# Get the LATEST checkpoint
latest_checkpoint = checkpoints[-1]  # checkpoint_01496.pt

# Load it
checkpoint_data = torch.load(latest_checkpoint, map_location='cpu')

if isinstance(checkpoint_data, torch.Tensor):
    loaded_embeddings = checkpoint_data.numpy()
else:
    loaded_embeddings = checkpoint_data['embeddings'].numpy()

actual_processed_count = loaded_embeddings.shape[0]  # ACTUAL count, not filename count

print(f"\n✓ Latest checkpoint: {os.path.basename(latest_checkpoint)}")
print(f"✓ ACTUAL sequences completed: {actual_processed_count}")
print(f"✓ Loaded embeddings shape: {loaded_embeddings.shape}\n")

# Step 2: Calculate remaining sequences
total_sequences = len(all_seqs)
remaining_sequences = total_sequences - actual_processed_count

print(f"📊 PROGRESS:")
print(f"   Total sequences: {total_sequences}")
print(f"   Completed: {actual_processed_count} ({actual_processed_count*100//total_sequences}%)")
print(f"   Remaining: {remaining_sequences}")
print(f"   Estimated time: {remaining_sequences * 0.5 / 60:.1f} minutes (CPU)\n")

# Step 3: Get remaining sequences (from 1500 to 2390)
remaining_seq_strs = [s['sequence'] for s in all_seqs[actual_processed_count:]]
remaining_indices = list(range(actual_processed_count, total_sequences))

print(f"[CONTINUING EMBEDDING GENERATION]")
print(f"Batch size: {Config.BATCH_SIZE}")
print(f"Device: {Config.DEVICE}\n")

# Step 4: Generate ONLY the remaining embeddings
try:
    start_time = time.time()
    
    # Use generator to create embeddings for remaining sequences
    new_embeddings_list = []
    
    batch_converter = alphabet.get_batch_converter()
    
    with torch.no_grad():
        pbar = tqdm(range(0, len(remaining_seq_strs), Config.BATCH_SIZE),
                   desc="Generating embeddings",
                   total=(len(remaining_seq_strs) + Config.BATCH_SIZE - 1) // Config.BATCH_SIZE)
        
        for batch_start in pbar:
            batch_end = min(batch_start + Config.BATCH_SIZE, len(remaining_seq_strs))
            batch_seqs = remaining_seq_strs[batch_start:batch_end]
            
            # Truncate if needed
            batch_seqs = [s[:Config.MAX_SEQ_LEN] for s in batch_seqs]
            
            # Convert to tokens
            batch_labels = [f"seq_{i}" for i in range(len(batch_seqs))]
            batch_data = list(zip(batch_labels, batch_seqs))
            _, _, tokens = batch_converter(batch_data)
            tokens = tokens.to(Config.DEVICE)
            
            # Generate embeddings
            if Config.DEVICE.type == 'cuda':
                with torch.cuda.amp.autocast():
                    results = model(tokens, repr_layers=[33], return_contacts=False)
            else:
                results = model(tokens, repr_layers=[33], return_contacts=False)
            
            token_embeddings = results["representations"][33]
            
            # Average across tokens
            for j, seq in enumerate(batch_seqs):
                emb = token_embeddings[j, 1:len(seq)+1].mean(0)
                new_embeddings_list.append(emb.cpu().numpy())
    
    # Convert to array
    new_embeddings = np.array(new_embeddings_list, dtype=np.float32)
    elapsed = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"✓ NEW EMBEDDINGS GENERATED!")
    print(f"{'='*70}")
    print(f"  Generated: {len(new_embeddings)} sequences")
    print(f"  Time: {elapsed/60:.1f} minutes")
    print(f"  Speed: {len(new_embeddings)/elapsed:.1f} seq/sec")
    print(f"{'='*70}\n")
    
    # Step 5: Combine with loaded embeddings
    print("✓ Combining embeddings...")
    combined_embeddings = np.vstack([loaded_embeddings, new_embeddings])
    embeddings = combined_embeddings
    
    print(f"✓ Final embeddings shape: {embeddings.shape}")
    print(f"✓ Total size: {embeddings.nbytes / (1024**3):.2f} GB\n")
    
except KeyboardInterrupt:
    print("\n⚠️  Interrupted! Saving progress...")
    # Save what we have so far
    if 'new_embeddings_list' in locals() and len(new_embeddings_list) > 0:
        temp_emb = np.array(new_embeddings_list, dtype=np.float32)
        temp_combined = np.vstack([loaded_embeddings, temp_emb])
        temp_path = f"{checkpoint_dir}/partial_embeddings.npz"
        np.savez_compressed(temp_path, embeddings=temp_combined)
        print(f"✓ Partial progress saved to: {temp_path}")
    raise

except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    print(traceback.format_exc())
    raise

# Step 6: IMPORTANT - Save final result
print("[SAVING FINAL EMBEDDINGS]")
final_path = f"{checkpoint_dir}/all_embeddings_COMPLETE.npz"
np.savez_compressed(final_path, embeddings=embeddings)
print(f"✓ Final embeddings saved: {final_path}")
print(f"✓ Shape: {embeddings.shape}")
print(f"✓ Size: {embeddings.nbytes / (1024**3):.2f} GB\n")

print("="*70)
print("✅ EMBEDDING GENERATION COMPLETE!")
print("="*70)

RESUMING EMBEDDING GENERATION - SMART CONTINUATION

✓ Latest checkpoint: checkpoint_01496.pt
✓ ACTUAL sequences completed: 1500
✓ Loaded embeddings shape: (1500, 1280)

📊 PROGRESS:
   Total sequences: 2390
   Completed: 1500 (62%)
   Remaining: 890
   Estimated time: 7.4 minutes (CPU)

[CONTINUING EMBEDDING GENERATION]
Batch size: 4
Device: cpu



Generating embeddings:  26%|███▋          | 58/223 [1:41:59<4:50:09, 105.51s/it]



⚠️  Interrupted! Saving progress...
✓ Partial progress saved to: data/embeddings/partial_embeddings.npz


KeyboardInterrupt: 

In [15]:
# ============================================================
# SAVE CURRENT PROGRESS (24%) BEFORE LAPTOP SHUTS DOWN
# ============================================================

import numpy as np
import torch
import os

checkpoint_dir = 'data/embeddings'

print("="*70)
print("SAVING CURRENT PROGRESS")
print("="*70)

# Check what we have
if 'new_embeddings_list' in locals() and len(new_embeddings_list) > 0:
    print(f"\n✓ Found {len(new_embeddings_list)} new embeddings in memory")
    
    # Convert to array
    temp_new_emb = np.array(new_embeddings_list, dtype=np.float32)
    print(f"✓ New embeddings shape: {temp_new_emb.shape}")
    
    # Combine with loaded embeddings
    temp_combined = np.vstack([loaded_embeddings, temp_new_emb])
    print(f"✓ Combined shape: {temp_combined.shape}")
    
    # Total sequences processed
    total_processed = temp_combined.shape[0]
    print(f"✓ Total sequences processed: {total_processed}/2390 ({total_processed*100//2390}%)")
    
    # Save as checkpoint
    checkpoint_path = f"{checkpoint_dir}/checkpoint_PROGRESS_{total_processed:05d}.pt"
    torch.save(torch.from_numpy(temp_combined), checkpoint_path)
    
    print(f"\n✅ SAVED: {checkpoint_path}")
    print(f"✅ Size: {os.path.getsize(checkpoint_path) / (1024**3):.2f} GB")
    
    # Also save as backup NPZ
    npz_path = f"{checkpoint_dir}/embeddings_backup_{total_processed:05d}.npz"
    np.savez_compressed(npz_path, embeddings=temp_combined)
    print(f"✅ BACKUP: {npz_path}")
    
    print("\n" + "="*70)
    print(f"✓ Safe to shutdown! Progress saved at {total_processed} sequences")
    print("="*70)

else:
    print("⚠️  No new embeddings found in memory")
    print("This might mean the cell wasn't running")

SAVING CURRENT PROGRESS

✓ Found 232 new embeddings in memory
✓ New embeddings shape: (232, 1280)
✓ Combined shape: (1732, 1280)
✓ Total sequences processed: 1732/2390 (72%)

✅ SAVED: data/embeddings/checkpoint_PROGRESS_01732.pt
✅ Size: 0.01 GB
✅ BACKUP: data/embeddings/embeddings_backup_01732.npz

✓ Safe to shutdown! Progress saved at 1732 sequences


In [4]:
! pip install numpy

In [16]:
# ============================================================
# RESUME FROM SAVED PROGRESS - FIXED (NO NUMPY CONVERSION)
# ============================================================

import os
import torch
import numpy as np
import glob
from tqdm import tqdm

checkpoint_dir = 'data/embeddings'

print("="*70)
print("CHECKING FOR SAVED PROGRESS")
print("="*70)

# Find all progress checkpoints
progress_checkpoints = sorted(glob.glob(f"{checkpoint_dir}/checkpoint_PROGRESS_*.pt"))

if progress_checkpoints:
    latest = progress_checkpoints[-1]
    filename = os.path.basename(latest)
    
    print(f"\n✓ Found saved progress: {filename}")
    
    # Load it - KEEP AS TORCH TENSOR
    loaded_embeddings_tensor = torch.load(latest, map_location='cpu')
    
    print(f"✓ Loaded checkpoint as tensor")
    print(f"✓ Type: {type(loaded_embeddings_tensor)}")
    print(f"✓ Shape: {loaded_embeddings_tensor.shape}")
    
    actual_processed_count = loaded_embeddings_tensor.shape[0]
    total_sequences = 2390
    remaining_sequences = total_sequences - actual_processed_count
    
    print(f"✓ Progress: {actual_processed_count}/{total_sequences} ({actual_processed_count*100//total_sequences}%)")
    print(f"✓ Remaining: {remaining_sequences} sequences\n")
    
    # Get remaining sequences
    remaining_seq_strs = [s['sequence'] for s in all_seqs[actual_processed_count:]]
    
    print(f"[RESUMING EMBEDDING GENERATION]")
    print(f"Batch size: {Config.BATCH_SIZE}")
    print(f"Device: {Config.DEVICE}\n")
    
    # ============================================================
    # CONTINUE GENERATION WITH CHECKPOINTS (KEEP AS TENSORS)
    # ============================================================
    
    try:
        start_time = time.time()
        new_embeddings_list = []
        batch_converter = alphabet.get_batch_converter()
        
        with torch.no_grad():
            pbar = tqdm(range(0, len(remaining_seq_strs), Config.BATCH_SIZE),
                       desc="Generating embeddings",
                       total=(len(remaining_seq_strs) + Config.BATCH_SIZE - 1) // Config.BATCH_SIZE)
            
            for batch_idx, batch_start in enumerate(pbar):
                batch_end = min(batch_start + Config.BATCH_SIZE, len(remaining_seq_strs))
                batch_seqs = remaining_seq_strs[batch_start:batch_end]
                
                # Truncate if needed
                batch_seqs = [s[:Config.MAX_SEQ_LEN] for s in batch_seqs]
                
                # Convert to tokens
                batch_labels = [f"seq_{i}" for i in range(len(batch_seqs))]
                batch_data = list(zip(batch_labels, batch_seqs))
                _, _, tokens = batch_converter(batch_data)
                tokens = tokens.to(Config.DEVICE)
                
                # Generate embeddings
                if Config.DEVICE.type == 'cuda':
                    with torch.cuda.amp.autocast():
                        results = model(tokens, repr_layers=[33], return_contacts=False)
                else:
                    results = model(tokens, repr_layers=[33], return_contacts=False)
                
                token_embeddings = results["representations"][33]
                
                # Average across tokens - KEEP AS TENSOR
                for j, seq in enumerate(batch_seqs):
                    emb = token_embeddings[j, 1:len(seq)+1].mean(0)
                    new_embeddings_list.append(emb.cpu().detach())  # Keep as tensor
                
                # Save checkpoint every 10 batches
                if (batch_idx + 1) % 10 == 0:
                    # Combine tensors
                    temp_new_emb = torch.stack(new_embeddings_list)  # Stack tensors
                    temp_combined = torch.vstack([loaded_embeddings_tensor, temp_new_emb])
                    
                    total_so_far = actual_processed_count + len(new_embeddings_list)
                    checkpoint_path = f"{checkpoint_dir}/checkpoint_RESUME_{total_so_far:05d}.pt"
                    torch.save(temp_combined, checkpoint_path)
                    
                    pbar.set_postfix({
                        'checkpoint': f'saved ({total_so_far}/{total_sequences})',
                        'elapsed': f'{(time.time() - start_time)/60:.1f}m'
                    })
        
        # Final combination - KEEP AS TENSOR
        new_embeddings = torch.stack(new_embeddings_list)
        combined_embeddings = torch.vstack([loaded_embeddings_tensor, new_embeddings])
        embeddings = combined_embeddings  # Keep as tensor
        
        elapsed = time.time() - start_time
        
        print(f"\n{'='*70}")
        print(f"✓ EMBEDDINGS COMPLETE!")
        print(f"{'='*70}")
        print(f"  Final shape: {embeddings.shape}")
        print(f"  Total time: {elapsed/60:.1f} minutes")
        print(f"  Speed: {len(remaining_seq_strs)/elapsed:.1f} seq/sec")
        print(f"{'='*70}\n")
        
        # Save final as tensor (no numpy needed!)
        final_path = f"{checkpoint_dir}/all_embeddings_COMPLETE.pt"
        torch.save(embeddings, final_path)
        print(f"✓ Final saved: {final_path}")
        print(f"✓ Size: {embeddings.element_size() * embeddings.nelement() / (1024**3):.2f} GB\n")
        
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted again!")
        if 'new_embeddings_list' in locals() and len(new_embeddings_list) > 0:
            temp_emb = torch.stack(new_embeddings_list)
            temp_combined = torch.vstack([loaded_embeddings_tensor, temp_emb])
            total_so_far = actual_processed_count + len(new_embeddings_list)
            
            checkpoint_path = f"{checkpoint_dir}/checkpoint_PROGRESS_{total_so_far:05d}.pt"
            torch.save(temp_combined, checkpoint_path)
            print(f"✓ Progress saved: {checkpoint_path}")
            print(f"✓ Total embeddings: {temp_combined.shape[0]}")
        raise

    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        print(traceback.format_exc())
        raise

else:
    print("⚠️  No saved progress found!")
    print("Need to run initial embedding generation first")

CHECKING FOR SAVED PROGRESS

✓ Found saved progress: checkpoint_PROGRESS_01732.pt
✓ Loaded checkpoint as tensor
✓ Type: <class 'torch.Tensor'>
✓ Shape: torch.Size([1732, 1280])
✓ Progress: 1732/2390 (72%)
✓ Remaining: 658 sequences

[RESUMING EMBEDDING GENERATION]
Batch size: 4
Device: cpu



Generating embeddings: 100%|█| 165/165 [4:37:15<00:00, 100.82s/it, checkpoint=sa



✓ EMBEDDINGS COMPLETE!
  Final shape: torch.Size([2390, 1280])
  Total time: 277.3 minutes
  Speed: 0.0 seq/sec

✓ Final saved: data/embeddings/all_embeddings_COMPLETE.pt
✓ Size: 0.01 GB

